# Heston FNO — Results, Evaluation & Pricing Surfaces

This notebook evaluates the trained Heston Fourier Neural Operator (FNO) checkpoint and
creates the figures needed for the project report.

In [2]:
# ============================================================
# FIX: Make the local heston_fno package visible to Python
# ============================================================

import os
import sys
from pathlib import Path

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")
SRC = REPO / "src"

# Check that the repository exists
print("Repository exists:", REPO.exists())
print("Repository path:", REPO)

# Check that the Python package exists
print("src exists:", SRC.exists())
print("heston_fno exists:", (SRC / "heston_fno").exists())

# Make src/ visible to Python
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# Move the notebook into the repository
os.chdir(REPO)

print("Working directory:", os.getcwd())
print("src in sys.path:", str(SRC) in sys.path)

# Now test the import
from heston_fno.data.dataset import HestonDataset
from heston_fno.data.preprocessing import split_indices
from heston_fno.benchmarking.monte_carlo import price_heston_call_mc_curve

print("\nSUCCESS: heston_fno imports correctly.")

Repository exists: True
Repository path: /content/FNO-for-Heston-Pricing-Engine-
src exists: False
heston_fno exists: False
Working directory: /content/FNO-for-Heston-Pricing-Engine-
src in sys.path: True


ModuleNotFoundError: No module named 'heston_fno'

In [1]:
import time
import torch
import numpy as np
import matplotlib.pyplot as plt

from heston_fno.data.dataset import HestonDataset
from heston_fno.data.preprocessing import split_indices
from heston_fno.benchmarking.monte_carlo import price_heston_call_mc_curve

ModuleNotFoundError: No module named 'heston_fno'

## 1. Colab setup — use the same paths as `colab_train(1).ipynb`

This notebook intentionally uses the exact Colab repository location and Drive/data layout from your training notebook.

In [ ]:
# ============================================================
# STEP 1 — Clone repo + install project
# Same setup as colab_train(1).ipynb
# ============================================================

import os
import sys
import math
import shutil
from pathlib import Path

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")
ROOT = REPO

if not REPO.exists():
    !git clone https://github.com/bananapants-87/FNO-for-Heston-Pricing-Engine-.git

os.chdir(REPO)

# Same install approach as the training notebook.
!pip install -q neuraloperator pyyaml
!pip install -q -e . --no-deps

print("Working directory:", os.getcwd())
print("Repository exists:", REPO.exists())

# ============================================================
# STEP 2 — Mount Drive and copy the same four data files
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = REPO / "data" / "final"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Shared evaluation settings used by later cells.
RESULTS_DIR = REPO / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EVAL_MAX_SAMPLES = None  # Set an integer for a faster smoke test.
SURFACE_TEST_POSITION = 0
FNO_BENCHMARK_BATCH_SIZE = 4
FNO_WARMUP = 5
FNO_REPEATS = 20
# Shared baseline comparison settings.  These must match the contract used
# to generate targets.pt; update them if your data used different values.
# The comparison always draws from the exact deterministic training indices
# actually used by train_fno.py (first max_train_samples after splitting).
COMPARISON_MAX_TRAIN_SAMPLES = 5  # None evaluates every trained sample; PDE/MC can be slow.
RUN_BASELINE_COMPARISON = True
OPTION_STRIKE = 100.0
OPTION_RATE = 0.05
OPTION_MATURITY = 1.0
ADI_N_STEPS = 100
MC_N_PATHS = 2000  # Must be even. Increase for tighter Monte Carlo confidence intervals.
MC_N_STEPS = 100
MC_SEED = 42

files = ["inputs.pt", "targets.pt", "s_grid.pt", "v_grid.pt"]

candidates = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/MyDrive/final"),
    Path("/content/final"),
    Path("/content"),
]

for fname in files:
    destination = DATA_DIR / fname

    if destination.exists():
        print(fname, "already present")
        continue

    found = False
    for location in candidates:
        source = location / fname
        if source.exists():
            shutil.copy2(source, destination)
            print("Copied:", fname, "from", source)
            found = True
            break

    if not found:
        print("MISSING:", fname)

print("\n--- DATA VERIFICATION ---")
for fname in files:
    p = DATA_DIR / fname
    print(f"{fname:12} ->", "FOUND" if p.exists() else "MISSING")

# ============================================================
# FIND / RESTORE THE TRAINED CHECKPOINT
# ============================================================

from pathlib import Path
import shutil

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")
CHECKPOINT_PATH = REPO / "checkpoints" / "heston_fno.pth"

# Possible locations for the trained checkpoint
checkpoint_candidates = [
    Path("/content/drive/MyDrive/FNO-Heston-Project/"
         "run_002_5000train_1000val_10epochs/heston_fno.pth"),

    CHECKPOINT_PATH,
    Path("/content/heston_fno.pth"),
    Path("/content/drive/MyDrive/heston_fno.pth"),
    Path("/content/drive/MyDrive/checkpoints/heston_fno.pth"),
    Path("/content/drive/MyDrive/FNO-for-Heston-Pricing-Engine-/"
         "checkpoints/heston_fno.pth"),
]

# Make sure the destination folder exists
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Find the first existing checkpoint
found_checkpoint = None

for candidate in checkpoint_candidates:
    if candidate.exists():
        found_checkpoint = candidate
        break

if found_checkpoint is None:
    raise FileNotFoundError(
        "No heston_fno.pth was found.\n\n"
        "Your training must finish first and create:\n"
        "/content/FNO-for-Heston-Pricing-Engine-/checkpoints/heston_fno.pth\n\n"
        "Or save the checkpoint to Google Drive, for example:\n"
        "/content/drive/MyDrive/heston_fno.pth"
    )

# Copy it into the location expected by results.ipynb
if found_checkpoint != CHECKPOINT_PATH:
    shutil.copy2(found_checkpoint, CHECKPOINT_PATH)
    print("Checkpoint copied from:")
    print(found_checkpoint)
    print("\nTo:")
    print(CHECKPOINT_PATH)
else:
    print("Checkpoint already exists at:")
    print(CHECKPOINT_PATH)

print("\nCheckpoint size:",
      round(CHECKPOINT_PATH.stat().st_size / (1024**2), 2),
      "MB")

print("READY — you can now run the next results notebook cell.")

In [ ]:
# ============================================================
# SETUP PATHS BEFORE IMPORTING heston_fno
# ============================================================

from pathlib import Path
import sys
import os
import subprocess

REPO = Path("/content/FNO-for-Heston-Pricing-Engine-")

# Make sure the repo exists
if not REPO.exists():
    !git clone https://github.com/bananapants-87/FNO-for-Heston-Pricing-Engine-.git

# Move into the project directory
os.chdir(REPO)

# Add src/ so "from heston_fno ..." works
SRC = REPO / "src"

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Working directory:", Path.cwd())
print("SRC path:", SRC)
print("src exists:", SRC.exists())

# Install project dependencies
!pip install -q neuraloperator pyyaml
!pip install -q -e . --no-deps

# ============================================================
# NOW import your project modules
# ============================================================

import torch
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader

from heston_fno.data.dataset import HestonDataset
from heston_fno.data.preprocessing import split_indices
from heston_fno.data.normalization import StandardScaler
from heston_fno.models.fno import build_model_from_config
from heston_fno.evaluation.errors import summarize_errors
from heston_fno.evaluation.greeks import (
    compute_delta,
    compute_vega,
    compute_reference_delta,
    compare_greeks,
)
from heston_fno.benchmarking.fno import benchmark_fno

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch:", torch.__version__)
print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Load the trained checkpoint, saved configuration and scalers

The checkpoint is the file produced by `scripts/train_fno.py` at:

`/content/FNO-for-Heston-Pricing-Engine-/checkpoints/heston_fno.pth`

The training script stores the model state, configuration, training history, input scaler and target scaler in this file.

In [ ]:
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False,
)

saved_config = checkpoint.get("config", None)
if saved_config is None:
    with open(ROOT / "configs" / "fno.yaml", "r", encoding="utf-8") as f:
        saved_config = yaml.safe_load(f) or {}

model = build_model_from_config(saved_config).to(device)

state_dict = checkpoint["model_state_dict"]
state_dict = {k: v for k, v in state_dict.items() if k != "_metadata"}
model.load_state_dict(state_dict)
model.eval()

input_scaler = StandardScaler()
target_scaler = StandardScaler()

if "input_scaler" in checkpoint and "target_scaler" in checkpoint:
    input_scaler.load_state_dict(checkpoint["input_scaler"])
    target_scaler.load_state_dict(checkpoint["target_scaler"])
else:
    raise KeyError(
        "Checkpoint does not contain input_scaler/target_scaler. "
        "Use a checkpoint created by the current train_fno.py."
    )

history = checkpoint.get("history", {})

print("Checkpoint loaded successfully.")
print("Saved epoch:", checkpoint.get("epoch", "not stored"))
print("Best validation loss:", checkpoint.get("best_val_loss", "not stored"))
print("Saved model config:", saved_config.get("model", {}))
print("History keys:", list(history.keys()))

In [ ]:
print("========== CHECKPOINT VERIFICATION ==========")

saved_config = checkpoint.get("config", {})
training_cfg = saved_config.get("training", {})
history = checkpoint.get("history", {})

print("Checkpoint:", found_checkpoint)
print("Saved epoch:", checkpoint.get("epoch"))
print("Number of epochs in history:", len(history.get("train_loss", [])))
print("Training samples:", training_cfg.get("max_train_samples"))
print("Validation samples:", training_cfg.get("max_val_samples"))
print("Batch size:", training_cfg.get("batch_size"))
print("Learning rate:", training_cfg.get("lr"))

print("==============================================")

## 3. Load the complete dataset and recreate the exact deterministic split

The repo uses an 80/10/10 split with seed `42` in the current configuration.

We use the raw dataset here, then apply the scaler stored in the trained checkpoint.
This avoids accidentally fitting new normalization statistics during evaluation.

In [ ]:
full_dataset = HestonDataset(
    inputs_path=DATA_DIR / "inputs.pt",
    targets_path=DATA_DIR / "targets.pt",
    s_grid_path=DATA_DIR / "s_grid.pt",
    v_grid_path=DATA_DIR / "v_grid.pt",
)

split_cfg = saved_config.get("training", {}).get("split", [0.8, 0.1, 0.1])
seed = saved_config.get("training", {}).get("seed", 42)

train_idx, val_idx, test_idx = split_indices(
    len(full_dataset),
    split=split_cfg,
    seed=seed,
)

print("Total samples:", len(full_dataset))
print("Train samples:", len(train_idx))
print("Validation samples:", len(val_idx))
print("Test samples:", len(test_idx))
print("Input sample shape:", tuple(full_dataset[0][0].shape))
print("Target sample shape:", tuple(full_dataset[0][1].shape))
print("S grid shape:", tuple(full_dataset.s_grid.shape))
print("v grid shape:", tuple(full_dataset.v_grid.shape))

In [ ]:
class ScaledIndexDataset(Dataset):
    def __init__(self, base_dataset, indices, input_scaler, target_scaler):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.input_scaler = input_scaler
        self.target_scaler = target_scaler

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        idx = self.indices[i]
        x_raw, y_raw = self.base_dataset[idx]
        x = self.input_scaler.transform(x_raw)
        y = self.target_scaler.transform(y_raw)
        return x, y

test_indices_used = list(test_idx)
if EVAL_MAX_SAMPLES is not None:
    test_indices_used = test_indices_used[: int(EVAL_MAX_SAMPLES)]

test_scaled_dataset = ScaledIndexDataset(
    full_dataset,
    test_indices_used,
    input_scaler,
    target_scaler,
)

eval_batch_size = int(
    saved_config.get("training", {}).get("batch_size", 4)
)

test_loader = DataLoader(
    test_scaled_dataset,
    batch_size=eval_batch_size,
    shuffle=False,
    num_workers=0,
)

print("Samples used for evaluation:", len(test_scaled_dataset))
print("Evaluation batch size:", eval_batch_size)

## 4. Evaluate the held-out test set

These metrics are computed in **physical option-price units** after inverse normalization.

The per-point mean relative error is retained for consistency with `scripts/evaluate.py`,
but it can become very large near target prices close to zero. The relative L2 error is
also reported because it is less sensitive to individual near-zero grid points.

In [ ]:
model.eval()

sum_sq = 0.0
sum_abs = 0.0
sum_rel = 0.0
sum_y = 0.0
sum_y2 = 0.0
sum_count = 0
max_abs = 0.0

with torch.no_grad():
    for x_scaled, y_scaled in test_loader:
        x_scaled = x_scaled.to(device)
        y_scaled = y_scaled.to(device)

        pred_scaled = model(x_scaled)

        pred = target_scaler.inverse_transform(pred_scaled)
        y = target_scaler.inverse_transform(y_scaled)

        diff = pred - y
        abs_diff = torch.abs(diff)

        sum_sq += torch.sum(diff ** 2).item()
        sum_abs += torch.sum(abs_diff).item()
        sum_rel += torch.sum(abs_diff / (torch.abs(y) + 1e-6)).item()

        sum_y += torch.sum(y).item()
        sum_y2 += torch.sum(y ** 2).item()
        sum_count += y.numel()

        batch_max = abs_diff.max().item()
        max_abs = max(max_abs, batch_max)

rmse_value = math.sqrt(sum_sq / max(sum_count, 1))
mae_value = sum_abs / max(sum_count, 1)
mre_value = sum_rel / max(sum_count, 1)

relative_l2_value = math.sqrt(sum_sq) / (math.sqrt(sum_y2) + 1e-7)

# Global R² over all price-grid points.
mean_y = sum_y / max(sum_count, 1)
sst = sum_y2 - 2.0 * mean_y * sum_y + sum_count * (mean_y ** 2)
r2_value = 1.0 - (sum_sq / sst) if sst > 0 else float("nan")

metrics = {
    "RMSE": rmse_value,
    "MAE": mae_value,
    "Maximum Absolute Error": max_abs,
    "Mean Relative Error": mre_value,
    "Relative L2 Error": relative_l2_value,
    "R2": r2_value,
}

metrics_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in metrics.items()]
)

display(metrics_df)

metrics_df.to_csv(RESULTS_DIR / "evaluation_metrics.csv", index=False)

print("\nMetrics saved to:", RESULTS_DIR / "evaluation_metrics.csv")

## 5. Training and validation loss curve

In [ ]:
train_loss = history.get("train_loss", [])
val_loss = history.get("val_loss", [])

if len(train_loss) == 0:
    print("No training history found in the checkpoint.")
else:
    epochs = np.arange(1, len(train_loss) + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss, marker="o", label="Training loss")
    if len(val_loss) == len(train_loss):
        plt.plot(epochs, val_loss, marker="o", label="Validation loss")

    plt.xlabel("Epoch")
    plt.ylabel("MSE loss (normalized target)")
    plt.title("FNO Training and Validation Loss")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "training_validation_loss.png", dpi=200)
    plt.show()

## 6. Select one held-out test sample and predict its full pricing surface

In [ ]:
if SURFACE_TEST_POSITION < 0 or SURFACE_TEST_POSITION >= len(test_idx):
    raise IndexError(
        f"SURFACE_TEST_POSITION must be between 0 and {len(test_idx)-1}."
    )

surface_raw_index = int(test_idx[SURFACE_TEST_POSITION])

x_raw, y_raw = full_dataset[surface_raw_index]

x_scaled = input_scaler.transform(x_raw).unsqueeze(0)
y_scaled = target_scaler.transform(y_raw).unsqueeze(0)

with torch.no_grad():
    pred_scaled = model(x_scaled.to(device))

pred_surface = target_scaler.inverse_transform(pred_scaled).cpu()[0]
true_surface = y_raw.unsqueeze(0).cpu()[0]

abs_error_surface = torch.abs(pred_surface - true_surface)

s_grid = full_dataset.s_grid.cpu()
v_grid = full_dataset.v_grid.cpu()

S, V = torch.meshgrid(s_grid, v_grid, indexing="ij")
S_np = S.numpy()
V_np = V.numpy()

pred_np = pred_surface.squeeze(0).numpy()
true_np = true_surface.squeeze(0).numpy()
err_np = abs_error_surface.squeeze(0).numpy()

print("Original dataset sample index:", surface_raw_index)
print("Parameter values [kappa, theta, sigma, rho, v0]:")
print(full_dataset.params[surface_raw_index].numpy())
print("Surface shape:", pred_surface.shape)
print("True surface min/max:", true_surface.min().item(), true_surface.max().item())
print("Predicted surface min/max:", pred_surface.min().item(), pred_surface.max().item())

## 7. Ground-truth pricing surface

This is the reference pricing surface stored in `targets.pt`.
In your project these are the prices generated by the reference PDE/ADI workflow used to
construct the dataset.

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, true_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Option price")
ax.set_title("Ground-Truth Heston Pricing Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Price")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "ground_truth_pricing_surface_3d.png", dpi=200)
plt.show()

## 8. FNO predicted pricing surface

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, pred_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Option price")
ax.set_title("FNO Predicted Heston Pricing Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Price")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_predicted_pricing_surface_3d.png", dpi=200)
plt.show()

## 9. Absolute-error surface

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, err_np, cmap="magma", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Absolute error")
ax.set_title("|FNO prediction − Ground truth|")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Absolute error")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "absolute_error_surface_3d.png", dpi=200)
plt.show()

## 10. Predicted-vs-true price comparison

This checks how closely individual grid-point predictions follow the reference prices.

In [ ]:
true_flat = true_np.ravel()
pred_flat = pred_np.ravel()

fig = plt.figure(figsize=(7, 6))
plt.scatter(true_flat, pred_flat, s=8, alpha=0.35)

lo = min(true_flat.min(), pred_flat.min())
hi = max(true_flat.max(), pred_flat.max())
plt.plot([lo, hi], [lo, hi], linestyle="--", label="Perfect prediction")

plt.xlabel("Ground-truth option price")
plt.ylabel("FNO predicted option price")
plt.title("FNO Prediction vs Ground Truth")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "prediction_vs_ground_truth.png", dpi=200)
plt.show()

## 11. Absolute-error histogram

In [ ]:
fig = plt.figure(figsize=(8, 5))
plt.hist(np.abs(pred_flat - true_flat), bins=50)
plt.xlabel("Absolute error")
plt.ylabel("Number of grid points")
plt.title("Distribution of Absolute Pricing Error")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "absolute_error_histogram.png", dpi=200)
plt.show()

## 12. Pricing cross-section at one variance level

This gives a conventional 2D pricing curve:

`S → option price`

for a fixed variance-grid index.

In [ ]:
v_index = len(v_grid) // 2
v_value = v_grid[v_index].item()

fig = plt.figure(figsize=(8, 5))
plt.plot(s_grid.numpy(), true_np[:, v_index], marker="o", markersize=3, label="Ground truth")
plt.plot(s_grid.numpy(), pred_np[:, v_index], marker="x", markersize=3, label="FNO")

plt.xlabel("Spot price S")
plt.ylabel("Option price")
plt.title(f"Pricing Cross-Section at variance v = {v_value:.6g}")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / "pricing_cross_section.png", dpi=200)
plt.show()

## 13. Error summary for the selected surface

In [ ]:
surface_summary = summarize_errors(
    pred_surface.unsqueeze(0),
    true_surface.unsqueeze(0),
).to_dict()

surface_summary_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in surface_summary.items()]
)
display(surface_summary_df)
surface_summary_df.to_csv(RESULTS_DIR / "selected_surface_metrics.csv", index=False)

## 14. Greeks / sensitivities

The repository provides:

- `compute_delta()` — derivative of the FNO output with respect to the spot-price `S` input channel.
- `compute_vega()` — derivative with respect to the initial variance `v0` input channel, so this is best
  described here as **Vega-like / v0 sensitivity**.

A reference Delta can be obtained directly from the ground-truth pricing surface by numerical differentiation.

The repository's `compute_reference_vega()` differentiates the pricing surface with respect to the
grid coordinate `v`; that is **not the same derivative** as `dV/dv0`, so this notebook does not
present those two quantities as a direct Vega-vs-reference comparison.

In [ ]:
# FNO Delta and v0-sensitivity for the selected sample.
delta_surface = compute_delta(
    model=model,
    x=x_scaled,
    input_scaler=input_scaler,
    target_scaler=target_scaler,
    s_channel=5,
    device=device,
)[0]

vega_like_surface = compute_vega(
    model=model,
    x=x_scaled,
    input_scaler=input_scaler,
    target_scaler=target_scaler,
    v0_channel=4,
    device=device,
)[0]

reference_delta = compute_reference_delta(
    true_surface,
    s_grid,
)[0]

delta_metrics = compare_greeks(
    delta_surface,
    reference_delta,
)

print("Delta comparison metrics:")
display(pd.DataFrame([{"Metric": k, "Value": v} for k, v in delta_metrics.items()]))

pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in delta_metrics.items()]
).to_csv(RESULTS_DIR / "delta_comparison_metrics.csv", index=False)

### Delta surface

In [ ]:
delta_np = delta_surface.squeeze(0).numpy()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, delta_np, cmap="viridis", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("Delta-like sensitivity")
ax.set_title("FNO Delta-like Sensitivity Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="Delta")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_delta_surface_3d.png", dpi=200)
plt.show()

### Vega-like / v0-sensitivity surface

In [ ]:
vega_like_np = vega_like_surface.squeeze(0).numpy()

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")
surf = ax.plot_surface(S_np, V_np, vega_like_np, cmap="plasma", linewidth=0, antialiased=True)
ax.set_xlabel("Spot price S")
ax.set_ylabel("Variance v")
ax.set_zlabel("dPrice/dv0")
ax.set_title("FNO Vega-like (Initial Variance Sensitivity) Surface")
fig.colorbar(surf, ax=ax, shrink=0.65, pad=0.1, label="dPrice/dv0")
fig.tight_layout()
fig.savefig(RESULTS_DIR / "fno_vega_like_surface_3d.png", dpi=200)
plt.show()

## 15. FNO inference-time benchmark

This uses the repository's timing helper with warm-up runs and repeated inference.

In [ ]:
bench_n = min(FNO_BENCHMARK_BATCH_SIZE, len(test_scaled_dataset))
bench_x = torch.stack([test_scaled_dataset[i][0] for i in range(bench_n)], dim=0)

fno_benchmark_result = benchmark_fno(
    model=model,
    batch=bench_x,
    device=device,
    warmup=FNO_WARMUP,
    repeats=FNO_REPEATS,
)

fno_time = fno_benchmark_result.average_time
fno_throughput = fno_benchmark_result.throughput

fno_results_df = pd.DataFrame([
    {"Metric": "FNO average inference time (s/batch)", "Value": fno_time},
    {"Metric": "FNO throughput (samples/s)", "Value": fno_throughput},
])

display(fno_results_df)
fno_results_df.to_csv(RESULTS_DIR / "fno_benchmark.csv", index=False)

## 16. Matched FNO, ADI, and Monte Carlo comparison on training samples

This comparison uses the same deterministic training indices that were passed to `train_fno.py`:
the first `max_train_samples` entries of `train_idx`. All three methods are evaluated against the
same stored reference surfaces. `COMPARISON_MAX_TRAIN_SAMPLES` limits that trained set for a
practical notebook runtime; set it to `None` to process every sample used in training.

ADI computes a full surface per parameter set. Monte Carlo computes one spot-price curve at each
variance-grid value, then stacks those curves into a surface. The strike, rate, and maturity must
match the data-generation contract for the accuracy metrics to be meaningful.

In [ ]:
from heston_fno.benchmarking.adi import solve_heston_adi
from heston_fno.benchmarking.monte_carlo import price_heston_call_mc_curve

def surface_metrics(prediction, reference):
    """Return global, physical-unit error metrics for two [N, H, W] arrays."""
    prediction = torch.as_tensor(prediction, dtype=torch.float32).cpu()
    reference = torch.as_tensor(reference, dtype=torch.float32).cpu()
    diff = prediction - reference
    sq_error = torch.sum(diff ** 2).item()
    count = diff.numel()
    reference_mean = reference.mean().item()
    total_sum_squares = torch.sum((reference - reference_mean) ** 2).item()
    return {
        "RMSE": math.sqrt(sq_error / max(count, 1)),
        "MAE": torch.mean(torch.abs(diff)).item(),
        "Relative L2 Error": math.sqrt(sq_error) / (torch.linalg.vector_norm(reference).item() + 1e-7),
        "R2": 1.0 - sq_error / total_sum_squares if total_sum_squares > 0 else float("nan"),
    }

training_limit = saved_config.get("training", {}).get("max_train_samples")
trained_indices = [int(index) for index in train_idx]
if training_limit is not None:
    trained_indices = trained_indices[: int(training_limit)]
comparison_indices = trained_indices
if COMPARISON_MAX_TRAIN_SAMPLES is not None:
    comparison_indices = comparison_indices[: int(COMPARISON_MAX_TRAIN_SAMPLES)]
if not comparison_indices:
    raise ValueError("No trained samples were selected for the baseline comparison.")

print(f"Training samples used by the checkpoint: {len(trained_indices)}")
print(f"Samples used for matched comparison: {len(comparison_indices)}")

raw_comparison_inputs = torch.stack([full_dataset[index][0] for index in comparison_indices])
reference_surfaces = torch.stack([full_dataset[index][1].squeeze(0) for index in comparison_indices])

# FNO: predict the same raw samples after applying the checkpoint scaler.
fno_comparison_input = input_scaler.transform(raw_comparison_inputs).to(device)
if device.type == "cuda":
    torch.cuda.synchronize()
fno_start = time.perf_counter()
with torch.no_grad():
    fno_surfaces = target_scaler.inverse_transform(model(fno_comparison_input)).cpu().squeeze(1)
if device.type == "cuda":
    torch.cuda.synchronize()
fno_comparison_time = time.perf_counter() - fno_start

if RUN_BASELINE_COMPARISON:
    # ADI and MC both consume raw Heston parameters, not normalized FNO channels.
    adi_start = time.perf_counter()
    adi_surfaces = torch.stack([
        solve_heston_adi(
            params=raw_x[:5, 0, 0], s_grid=s_grid, v_grid=v_grid,
            strike=OPTION_STRIKE, r=OPTION_RATE, maturity=OPTION_MATURITY, n_steps=ADI_N_STEPS,
        )
        for raw_x in raw_comparison_inputs
    ])
    adi_comparison_time = time.perf_counter() - adi_start

    mc_start = time.perf_counter()
    mc_surfaces, mc_standard_errors, mc_ci_lows, mc_ci_highs = [], [], [], []
    for sample_offset, raw_x in enumerate(raw_comparison_inputs):
        params = raw_x[:5, 0, 0]
        curve_results = [
            price_heston_call_mc_curve(
                params=params, s_grid=s_grid, v0=float(v0),
                strike=OPTION_STRIKE, r=OPTION_RATE, maturity=OPTION_MATURITY,
                n_paths=MC_N_PATHS, n_steps=MC_N_STEPS,
                seed=MC_SEED + sample_offset * len(v_grid) + v_offset, device=device,
            )
            for v_offset, v0 in enumerate(v_grid)
        ]
        mc_surfaces.append(torch.stack([result.price for result in curve_results], dim=1))
        mc_standard_errors.append(torch.stack([result.standard_error for result in curve_results], dim=1))
        mc_ci_lows.append(torch.stack([result.ci95_low for result in curve_results], dim=1))
        mc_ci_highs.append(torch.stack([result.ci95_high for result in curve_results], dim=1))
    mc_surfaces = torch.stack(mc_surfaces)
    mc_standard_errors = torch.stack(mc_standard_errors)
    mc_ci_lows = torch.stack(mc_ci_lows)
    mc_ci_highs = torch.stack(mc_ci_highs)
    mc_comparison_time = time.perf_counter() - mc_start

    method_outputs = {
        "FNO": (fno_surfaces, fno_comparison_time),
        "HV-ADI": (adi_surfaces, adi_comparison_time),
        "Monte Carlo": (mc_surfaces, mc_comparison_time),
    }
    rows = []
    for method, (surfaces, elapsed) in method_outputs.items():
        row = {
            "Method": method, "Samples": len(comparison_indices),
            "Runtime (s)": elapsed, "Throughput (samples/s)": len(comparison_indices) / elapsed,
            **surface_metrics(surfaces, reference_surfaces),
        }
        if method == "Monte Carlo":
            row["Mean MC standard error"] = mc_standard_errors.mean().item()
            row["Reference in MC 95% CI (%)"] = 100.0 * ((reference_surfaces >= mc_ci_lows) & (reference_surfaces <= mc_ci_highs)).float().mean().item()
        rows.append(row)
    comparison_results_df = pd.DataFrame(rows)
    adi_runtime = comparison_results_df.loc[comparison_results_df["Method"] == "HV-ADI", "Runtime (s)"].iloc[0]
    comparison_results_df["Speedup vs HV-ADI"] = adi_runtime / comparison_results_df["Runtime (s)"]

    display(comparison_results_df)
    comparison_results_df.to_csv(RESULTS_DIR / "trained_samples_method_comparison.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].bar(comparison_results_df["Method"], comparison_results_df["RMSE"])
    axes[0].set_title("Accuracy on matched training samples")
    axes[0].set_ylabel("RMSE")
    axes[1].bar(comparison_results_df["Method"], comparison_results_df["Throughput (samples/s)"])
    axes[1].set_yscale("log")
    axes[1].set_title("Performance on matched training samples")
    axes[1].set_ylabel("Samples / second (log scale)")
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "trained_samples_method_comparison.png", dpi=200)
    plt.show()
else:
    comparison_results_df = pd.DataFrame()
    print("FNO completed. Set RUN_BASELINE_COMPARISON = True to run ADI and Monte Carlo.")

In [ ]:
# ============================================================
# OPTIONAL — Save checkpoint to Google Drive for future sessions
# ============================================================

SAVE_CHECKPOINT_TO_DRIVE = True

if SAVE_CHECKPOINT_TO_DRIVE:
    drive_destination = Path("/content/drive/MyDrive/heston_fno.pth")
    shutil.copy2(CHECKPOINT_PATH, drive_destination)
    print("Checkpoint saved to:", drive_destination)
else:
    print("Checkpoint not copied to Drive. Set SAVE_CHECKPOINT_TO_DRIVE = True when desired.")

## 17. Save a compact report summary

In [ ]:
report_summary = {
    "device": str(device),
    "checkpoint": str(CHECKPOINT_PATH),
    "test_samples_evaluated": len(test_scaled_dataset),
    "surface_test_position": SURFACE_TEST_POSITION,
    "surface_original_dataset_index": surface_raw_index,
    "RMSE": rmse_value,
    "MAE": mae_value,
    "Maximum Absolute Error": max_abs,
    "Mean Relative Error": mre_value,
    "Relative L2 Error": relative_l2_value,
    "R2": r2_value,
    "FNO average inference time (s/batch)": fno_time,
    "FNO throughput (samples/s)": fno_throughput,
    "trained_samples_compared": len(comparison_indices),
    "baseline_comparison_ran": RUN_BASELINE_COMPARISON,
}

if not comparison_results_df.empty:
    for row in comparison_results_df.to_dict(orient="records"):
        method = row.pop("Method")
        for metric, value in row.items():
            report_summary[f"{method} {metric}"] = value

summary_df = pd.DataFrame(
    [{"Metric": k, "Value": v} for k, v in report_summary.items()]
)
display(summary_df)

with open(RESULTS_DIR / "results_summary.txt", "w", encoding="utf-8") as f:
    for key, value in report_summary.items():
        f.write(f"{key}: {value}\n")

print("\nAll result files saved under:")
print(RESULTS_DIR)

In [ ]:
# ============================================================
# SAVE ALL RESULTS TO GOOGLE DRIVE
# ============================================================

from google.colab import drive
from pathlib import Path
import shutil

# Mount Google Drive
drive.mount("/content/drive")

# Folder in Google Drive where you want to keep the results
DRIVE_RESULTS_DIR = Path(
    "/content/drive/MyDrive/Heston FNO Results"
)

DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Copy everything generated by results.ipynb
shutil.copytree(
    RESULTS_DIR,
    DRIVE_RESULTS_DIR,
    dirs_exist_ok=True
)

print("All results saved to:")
print(DRIVE_RESULTS_DIR)